In [1]:
import os
import time
import pickle
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import polars as pl
import scipy.stats
import pywt
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error
import xgboost as xgb
import lightgbm as lgb
from tqdm import tqdm

In [2]:
class Config:
    # Paths
    DATA_PATH = '/kaggle/input/ariel-data-challenge-2025'
    PREP_PATH = '/kaggle/input/ariel-data-challenge-2025-af-npy/'
    OUT_PATH  = '/kaggle/working/'
    # Files
    TRAIN_LABELS = os.path.join(DATA_PATH, 'train.csv')
    STAR_INFO    = os.path.join(DATA_PATH, 'train_star_info.csv')
    TEST_INFO    = os.path.join(DATA_PATH, 'test_star_info.csv')
    SAMPLE_SUB   = os.path.join(DATA_PATH, 'sample_submission.csv')
    WAVES        = os.path.join(DATA_PATH, 'wavelengths.csv')
    # CV
    N_FOLDS      = 5
    RANDOM_STATE = 42
    # Model params
    XGB_PARAMS = {
        'objective': 'reg:squarederror',
        'n_estimators': 1000,
        'learning_rate': 0.03,
        'max_depth': 6,
        'subsample': 0.8,
        'colsample_bytree': 0.7,
        'tree_method': 'hist',
        'random_state': RANDOM_STATE,
        'verbosity': 0
    }
    LGB_PARAMS = {
        'objective': 'regression',
        'learning_rate': 0.02,
        'num_leaves': 128,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'min_data_in_leaf': 30,
        'n_estimators': 2000,
        'verbose': -1,
        'seed': RANDOM_STATE
    }
    EARLY_STOPPING = 50

cfg = Config()

In [3]:
def load_raw(path, dataset, planet_ids, signal_file, normalize_factor):
    arr = np.zeros((len(planet_ids), normalize_factor.size), dtype=np.float32)
    for i, pid in enumerate(tqdm(planet_ids, desc=f"Loading {signal_file}")):
        df = pl.read_parquet(f"{path}/{dataset}/{int(pid)}/{signal_file}")
        sig = df.cast(pl.Int32).sum_horizontal().to_numpy().astype(np.float32)
        diff = sig[1::2] - sig[0::2]
        arr[i] = diff
    return arr

def make_features(f_raw, a_raw, star_info):
    N = f_raw.shape[1]
    transit = f_raw[:, 23500:44000]
    feats = {}
    # rolling mean/std on transit in windows of 1000
    window = 1000
    for i in range((transit.shape[1] // window)):
        seg = transit[:, i*window:(i+1)*window]
        feats[f'roll_mean_{i}'] = seg.mean(1)
        feats[f'roll_std_{i}']  = seg.std(1)
    # wavelength‑weighted means
    waves = pd.read_csv(cfg.WAVES)
    weights = waves.iloc[:, 1].to_numpy()
    feats['fgs_wmean'] = (transit * weights[:transit.shape[1]]).sum(1)/ weights[:transit.shape[1]].sum()
    # FFT & wavelet as before
    fft = np.fft.rfft(transit, axis=1)
    for k in range(1, 6):
        feats[f'fft_{k}'] = np.abs(fft[:, k])
    for lvl in range(1, 4):
        c = pywt.wavedec(transit, 'db4', level=lvl, axis=1)[0]
        feats[f'wav_std_{lvl}'] = c.std(1)
    # AIRS block as before
    a_tr = a_raw[:, 500:4500]
    feats['airs_mean'] = a_tr.mean(1)
    feats['airs_skew'] = scipy.stats.skew(a_tr, axis=1)
    fft_a = np.fft.rfft(a_tr, axis=1)
    for k in range(1, 6):
        feats[f'a_fft_{k}'] = np.abs(fft_a[:, k])
    # metadata
    meta = star_info.fillna(star_info.median())
    meta['Mp_P'] = meta['Mp']/(meta['P']+1e-6)
    meta['logRs_logTs'] = np.log1p(meta['Rs'])*np.log1p(meta['Ts'])
    X = pd.DataFrame(feats, index=star_info.index)
    X = pd.concat([X, meta], axis=1).fillna(0)
    return X

def official_score(y_true, y_pred, sigma, mu0, sigma0):
    y_true,y_pred,sigma = map(np.array, (y_true,y_pred,sigma))
    base = scipy.stats.norm.logpdf(y_true, loc=y_true, scale=sigma0)
    glp  = scipy.stats.norm.logpdf(y_true, loc=y_pred, scale=sigma)
    gm   = scipy.stats.norm.logpdf(y_true, loc=mu0,     scale=sigma0)
    return ((glp - gm)/(base - gm + 1e-9)).mean()

In [4]:
labels = pd.read_csv(cfg.TRAIN_LABELS, index_col='planet_id')
stars  = pd.read_csv(cfg.STAR_INFO,    index_col='planet_id').loc[labels.index]
f_raw  = np.load(cfg.PREP_PATH + 'f_raw_train.npy')
a_raw  = np.load(cfg.PREP_PATH + 'a_raw_train.npy')
X      = make_features(f_raw, a_raw, stars)
y      = labels.values
mu0, sigma0 = y.mean(), y.std()

groups = pd.qcut(stars['Rs'], cfg.N_FOLDS, labels=False).values
oof_med = np.zeros_like(y)
oof_sig = np.zeros_like(y)

kf = GroupKFold(n_splits=cfg.N_FOLDS)
for fold, (tr, val) in enumerate(kf.split(X, groups=groups), 1):
    print(f"Fold {fold}/{cfg.N_FOLDS}")
    X_tr, X_val = X.iloc[tr], X.iloc[val]
    y_tr, y_val = y[tr], y[val]

    # --- XGBoost with early stopping ---
    xgb_model = xgb.XGBRegressor(**cfg.XGB_PARAMS)
    xgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        early_stopping_rounds=cfg.EARLY_STOPPING,
        verbose=False
    )
    xgb_pred = xgb_model.predict(X_val)

    # --- LightGBM without early stopping in MultiOutputRegressor ---
    lgbm = MultiOutputRegressor(lgb.LGBMRegressor(**cfg.LGB_PARAMS))
    lgbm.fit(X_tr, y_tr)
    lgb_pred = lgbm.predict(X_val)

    # --- stacking for median ---
    stacked = np.stack([xgb_pred, lgb_pred], axis=2).mean(2)
    ridge = Ridge(random_state=cfg.RANDOM_STATE).fit(stacked, y_val)
    med = ridge.predict(stacked)
    oof_med[val] = med

    # --- raw sigma from quantile LightGBM (also without early stopping) ---
    lower = MultiOutputRegressor(
        lgb.LGBMRegressor(**{**cfg.LGB_PARAMS, 'objective':'quantile', 'alpha':0.05})
    ).fit(X_tr, y_tr).predict(X_val)
    upper = MultiOutputRegressor(
        lgb.LGBMRegressor(**{**cfg.LGB_PARAMS, 'objective':'quantile', 'alpha':0.95})
    ).fit(X_tr, y_tr).predict(X_val)
    oof_sig[val] = (upper - lower) / 3.29

Fold 1/5
Fold 2/5
Fold 3/5
Fold 4/5
Fold 5/5


In [5]:
calibrators = []
for dim in range(y.shape[1]):
    ir = IsotonicRegression(out_of_bounds='clip')
    ir.fit(oof_sig[:,dim], np.abs(y[:,dim]-oof_med[:,dim]))
    calibrators.append(ir)

with open(cfg.OUT_PATH + 'calibrators.pkl', 'wb') as f:
    pickle.dump(calibrators, f)
pickle.dump(X.columns.tolist(), open(cfg.OUT_PATH + 'features.pkl','wb'))

score = official_score(y, oof_med, np.column_stack([c.predict(oof_sig[:,d]) for d,c in enumerate(calibrators)]), mu0, sigma0)
print(f"\nOOF official score: {score:.6f}")

sample = pd.read_csv(cfg.SAMPLE_SUB, index_col='planet_id')
stars_test = pd.read_csv(cfg.TEST_INFO, index_col='planet_id')
f_test = load_raw(cfg.DATA_PATH, 'test', stars_test.index, 'FGS1_signal_0.parquet', np.arange(67500))
a_test = load_raw(cfg.DATA_PATH, 'test', stars_test.index, 'AIRS-CH0_signal_0.parquet', np.arange(5625))
X_test = make_features(f_test, a_test, stars_test)

xgb_full = xgb.XGBRegressor(**cfg.XGB_PARAMS).fit(X, y)
lgb_full = MultiOutputRegressor(lgb.LGBMRegressor(**cfg.LGB_PARAMS)).fit(X, y)
pred_x = xgb_full.predict(X_test)
pred_l = lgb_full.predict(X_test)
stack_full = np.stack([pred_x, pred_l], axis=2).mean(2)
ridge_full = Ridge(random_state=cfg.RANDOM_STATE).fit(
    np.stack([xgb_full.predict(X), lgb_full.predict(X)], axis=2).mean(2), y
)
med_full = ridge_full.predict(stack_full)

low_full = MultiOutputRegressor(
    lgb.LGBMRegressor(**{**cfg.LGB_PARAMS, 'objective':'quantile','alpha':0.05})
).fit(X, y).predict(X_test)
up_full  = MultiOutputRegressor(
    lgb.LGBMRegressor(**{**cfg.LGB_PARAMS, 'objective':'quantile','alpha':0.95})
).fit(X, y).predict(X_test)
raw_full = (up_full - low_full)/3.29

with open(cfg.OUT_PATH + 'calibrators.pkl','rb') as f:
    calibrators = pickle.load(f)
sig_full = np.column_stack([calibrators[d].predict(raw_full[:,d]) for d in range(raw_full.shape[1])])


OOF official score: 115417.656721


Loading AIRS-CH0_signal_0.parquet: 100%|██████████| 1/1 [00:01<00:00,  1.46s/it]


In [6]:
waves = pd.read_csv(cfg.WAVES)
df_med = pd.DataFrame(np.clip(med_full, 0, None), index=sample.index, columns=waves.columns)
df_sig = pd.DataFrame(sig_full, index=sample.index, columns=[f'sigma_{i+1}' for i in range(waves.shape[1])])
submission = pd.concat([df_med, df_sig], axis=1)
submission.to_csv('submission.csv')
print("Done! submission.csv created.")

Done! submission.csv created.
